In [2]:
import os
import sys
import clr
import pandas as pd


# ==========================================================
# 1. CONFIGURACIÓN DE LA CONSULTA
#    ESTA ES LA PARTE QUE TÚ MODIFICARÁS
# ==========================================================

TAGS = [
    "FCCPI13162M.PV",
    "FCCFI10561M.DACA.PV",
    "FCCFI30202.DACA.PV",
    "FCCTI30204.DACA.PV",
    "FCCZI10561.PV",
    "FCCZI10461.PV",
    "FCCSI70129B.PV",
    "FCCPDX12601A.DACA.PV",
]


# Rango de tiempo
INICIO = "NOW-6D"
FIN = "NOW"


# Intervalo de muestreo en minutos
INTERVALO_MIN = 30


# Tipo de muestreo:
# "Average"  = promedio de cada intervalo
# "Snapshot" = valor puntual de cada intervalo

TIPO_MUESTREO = "Average"


# ==========================================================
# 2. CONFIGURACIÓN DE UNIFORMANCE PHD
# ==========================================================

RUTA = r"C:\Program Files\Honeywell\Uniformance\Excel Companion"

if RUTA not in sys.path:
    sys.path.append(RUTA)

if hasattr(os, "add_dll_directory"):
    dll_dir = os.add_dll_directory(RUTA)

clr.AddReference(
    os.path.join(RUTA, "phdapinet.dll")
)


# ==========================================================
# 3. IMPORTAR CLASES DE UNIFORMANCE
# ==========================================================

from Uniformance.PHD import (
    PHDServer,
    PHDHistorian,
    Tag,
    SERVERVERSION,
    SAMPLETYPE,
    REDUCTIONTYPE,
    REDUCTIONOFFSET
)


# ==========================================================
# 4. CONECTAR AL SERVIDOR PHD
# ==========================================================

print("Conectando al PHD...")

server = PHDServer(
    "10.2.18.69",
    SERVERVERSION.RAPI200
)

server.ConnectToServer()

print("✅ Conectado al PHD")
print("Servidor:", server.HostName)
print("Puerto:", server.Port)
print("API:", server.APIVersion)


# ==========================================================
# 5. CONFIGURAR CONSULTA HISTÓRICA
# ==========================================================

historian = PHDHistorian()

historian.DefaultServer = server

historian.StartTime = INICIO
historian.EndTime = FIN


# Convertir minutos a segundos
historian.SampleFrequency = INTERVALO_MIN * 60

historian.UseSampleFrequency = True


# ==========================================================
# 6. DEFINIR TIPO DE MUESTREO
# ==========================================================

if TIPO_MUESTREO.lower() == "average":

    historian.Sampletype = SAMPLETYPE.Average

elif TIPO_MUESTREO.lower() == "snapshot":

    historian.Sampletype = SAMPLETYPE.Snapshot

else:

    raise ValueError(
        "TIPO_MUESTREO debe ser 'Average' o 'Snapshot'"
    )


print("\nCONFIGURACIÓN DE CONSULTA")
print("--------------------------")
print("Inicio:", INICIO)
print("Fin:", FIN)
print("Intervalo:", INTERVALO_MIN, "minutos")
print("Tipo:", TIPO_MUESTREO)
print("Número de tags:", len(TAGS))


# ==========================================================
# 7. FUNCIÓN PARA CONSULTAR UN TAG
# ==========================================================

def obtener_tag(nombre_tag):

    print(f"\nConsultando: {nombre_tag}")

    # Crear objeto Tag
    tag = Tag()

    tag.TagName = nombre_tag
    tag.Server = server


    # Sin reducción adicional
    tag.ReductionType = getattr(
        REDUCTIONTYPE,
        "None"
    )

    tag.ReductionFrequency = 0

    tag.ReductionOffset = REDUCTIONOFFSET.Before


    # ======================================================
    # CONSULTAR PHD
    # ======================================================

    datos = historian.FetchDataList(tag)


    # ======================================================
    # CONVERTIR DATOS .NET A PYTHON
    # ======================================================

    timestamps = list(datos.Timestamps)
    valores = list(datos.DataValues)


    # ======================================================
    # CREAR DATAFRAME DEL TAG
    # NO INCLUIMOS CONFIDENCE
    # ======================================================

    df_tag = pd.DataFrame({

        "Timestamp_PHD": timestamps,

        nombre_tag: valores

    })


    # ======================================================
    # CONVERTIR TIMESTAMP PHD A FECHA/HORA
    # ======================================================

    df_tag["FechaHora"] = pd.to_datetime(

        df_tag["Timestamp_PHD"],

        unit="D",

        origin="1899-12-30"

    ).dt.round("min")


    # ======================================================
    # ELIMINAR TIMESTAMP NUMÉRICO
    # ======================================================

    df_tag = df_tag.drop(
        columns=["Timestamp_PHD"]
    )


    # ======================================================
    # SI EXISTE MÁS DE UN DATO EN EL MISMO MINUTO
    # SE QUEDA CON EL PROMEDIO
    # ======================================================

    df_tag = (
        df_tag
        .groupby(
            "FechaHora",
            as_index=False
        )[nombre_tag]
        .mean()
    )


    # ======================================================
    # ORDENAR COLUMNAS
    # ======================================================

    df_tag = df_tag[
        [
            "FechaHora",
            nombre_tag
        ]
    ]


    print(
        f"✅ {len(df_tag)} datos obtenidos"
    )


    return df_tag


# ==========================================================
# 8. CONSULTAR TODOS LOS TAGS
# ==========================================================

dataframes = []


for nombre_tag in TAGS:

    try:

        df_tag = obtener_tag(nombre_tag)

        dataframes.append(df_tag)

    except Exception as e:

        print(
            f"\n❌ Error consultando {nombre_tag}"
        )

        print(e)


# ==========================================================
# 9. VERIFICAR QUE HAYA DATOS
# ==========================================================

if len(dataframes) == 0:

    raise RuntimeError(
        "No se pudo obtener información de ningún tag."
    )


# ==========================================================
# 10. UNIR TODOS LOS TAGS POR FECHA/HORA
# ==========================================================

df = dataframes[0]


for df_tag in dataframes[1:]:

    df = pd.merge(

        df,

        df_tag,

        on="FechaHora",

        how="outer"

    )


# ==========================================================
# 11. ORDENAR RESULTADOS
# ==========================================================

df = df.sort_values(
    "FechaHora"
).reset_index(drop=True)


# ==========================================================
# 12. MOSTRAR RESULTADO FINAL
# ==========================================================

print("\n")
print("=" * 70)
print("✅ CONSULTA PHD TERMINADA")
print("=" * 70)

print("\nCantidad de registros:", len(df))

print("\nDatos:\n")

print(
    df.to_string(
        index=False
    )
)


# ==========================================================
# 13. DATAFRAME FINAL
# ==========================================================

df

Conectando al PHD...
✅ Conectado al PHD
Servidor: 10.2.18.69
Puerto: 3150
API: RAPI200

CONFIGURACIÓN DE CONSULTA
--------------------------
Inicio: NOW-6D
Fin: NOW
Intervalo: 30 minutos
Tipo: Average
Número de tags: 8

Consultando: FCCPI13162M.PV
✅ 289 datos obtenidos

Consultando: FCCFI10561M.DACA.PV
✅ 289 datos obtenidos

Consultando: FCCFI30202.DACA.PV
✅ 289 datos obtenidos

Consultando: FCCTI30204.DACA.PV
✅ 289 datos obtenidos

Consultando: FCCZI10561.PV
✅ 289 datos obtenidos

Consultando: FCCZI10461.PV
✅ 289 datos obtenidos

Consultando: FCCSI70129B.PV
✅ 289 datos obtenidos

Consultando: FCCPDX12601A.DACA.PV
✅ 289 datos obtenidos


✅ CONSULTA PHD TERMINADA

Cantidad de registros: 289

Datos:

          FechaHora  FCCPI13162M.PV  FCCFI10561M.DACA.PV  FCCFI30202.DACA.PV  FCCTI30204.DACA.PV  FCCZI10561.PV  FCCZI10461.PV  FCCSI70129B.PV  FCCPDX12601A.DACA.PV
2026-08-11 00:26:00        0.909687          7982.488281            3.403623           27.083803      99.728943      99.658936 

,FechaHora,FCCPI13162M.PV,FCCFI10561M.DACA.PV,FCCFI30202.DACA.PV,FCCTI30204.DACA.PV,FCCZI10561.PV,FCCZI10461.PV,FCCSI70129B.PV,FCCPDX12601A.DACA.PV
0,2026-08-11 00:26:00,0.909687,7982.488281,3.403623,27.083803,99.728943,99.658936,0.000000,0.000000
1,2026-08-11 00:56:00,0.909194,8204.392578,3.394902,27.087120,99.725685,99.659744,0.000000,0.000000
2,2026-08-11 01:26:00,0.909015,8112.208984,3.343394,27.088688,99.732407,99.659744,0.000000,0.000466
3,2026-08-11 01:56:00,0.909035,8463.116211,3.097743,27.086369,99.749046,99.635963,0.000000,0.000000
4,2026-08-11 02:26:00,0.908995,8366.023438,2.968743,27.085890,99.747055,99.638550,0.000000,0.003157
...,...,...,...,...,...,...,...,...,...
284,2026-08-16 22:26:00,0.076144,99987.921875,3857.679443,20.059422,99.653236,42.212826,5325.729980,13.092964
285,2026-08-16 22:56:00,0.075528,100001.187500,3864.552490,20.011389,99.657303,42.282997,5327.231934,13.266320
286,2026-08-16 23:26:00,0.075032,99978.359375,3860.277344,20.007311,99.659142,42.226051,5326.808105,13.303896
287,2026-08-16 23:56:00,0.073812,99936.882812,3859.423340,19.996216,99.660355,42.202522,5326.796875,13.128271


In [3]:
df.head(20)

,FechaHora,FCCPI13162M.PV,FCCFI10561M.DACA.PV,FCCFI30202.DACA.PV,FCCTI30204.DACA.PV,FCCZI10561.PV,FCCZI10461.PV,FCCSI70129B.PV,FCCPDX12601A.DACA.PV
0,2026-08-11 00:26:00,0.909687,7982.488281,3.403623,27.083803,99.728943,99.658936,0.0,0.000000
1,2026-08-11 00:56:00,0.909194,8204.392578,3.394902,27.087120,99.725685,99.659744,0.0,0.000000
2,2026-08-11 01:26:00,0.909015,8112.208984,3.343394,27.088688,99.732407,99.659744,0.0,0.000466
3,2026-08-11 01:56:00,0.909035,8463.116211,3.097743,27.086369,99.749046,99.635963,0.0,0.000000
4,2026-08-11 02:26:00,0.908995,8366.023438,2.968743,27.085890,99.747055,99.638550,0.0,0.003157
5,2026-08-11 02:56:00,0.909422,8613.191406,3.166217,27.084507,99.741821,99.641060,0.0,0.000000
6,2026-08-11 03:26:00,0.908996,8275.720703,3.446237,27.092472,99.745361,99.646935,0.0,0.000000
7,2026-08-11 03:56:00,0.908440,8146.314453,3.341038,27.092148,99.747452,99.645065,0.0,0.000000
8,2026-08-11 04:26:00,0.908384,8350.382812,3.299591,27.090282,99.750099,99.663994,0.0,0.000000
9,2026-08-11 04:56:00,0.908531,8387.400391,3.370349,27.085121,99.745461,99.678467,0.0,0.000000


In [4]:
df.tail(20)

,FechaHora,FCCPI13162M.PV,FCCFI10561M.DACA.PV,FCCFI30202.DACA.PV,FCCTI30204.DACA.PV,FCCZI10561.PV,FCCZI10461.PV,FCCSI70129B.PV,FCCPDX12601A.DACA.PV
269,2026-08-16 14:56:00,0.093578,99984.429688,3878.069092,23.016863,99.495323,41.842880,5324.675781,13.158190
270,2026-08-16 15:26:00,0.094195,99979.992188,3870.604248,22.924059,99.502647,41.878559,5327.665039,13.141941
271,2026-08-16 15:56:00,0.093453,99948.671875,3882.483154,22.886087,99.525009,41.868782,5325.960938,13.048161
272,2026-08-16 16:26:00,0.089576,99945.265625,3884.957275,22.708450,99.540054,41.874359,5325.771973,13.107752
273,2026-08-16 16:56:00,0.089009,99976.328125,3890.533691,22.482800,99.557358,42.025742,5327.125488,12.931794
274,2026-08-16 17:26:00,0.086249,99980.406250,3891.554688,22.316029,99.578316,42.015026,5325.061035,13.199786
275,2026-08-16 17:56:00,0.086096,99925.554688,3893.865723,22.024839,99.595016,42.082150,5326.353516,13.049459
276,2026-08-16 18:26:00,0.081870,99964.914062,3896.811768,21.566921,99.606026,42.104862,5325.452637,13.282515
277,2026-08-16 18:56:00,0.079005,99962.500000,3897.740723,20.845367,99.617424,42.133244,5326.446289,13.154240
278,2026-08-16 19:26:00,0.075221,99920.906250,3893.115234,20.540188,99.627541,42.127415,5325.725586,13.207577
